# Interaction features (products of features)

_Feature Engineering_

**Multiply two features so a linear model can say 'the effect of one depends on the other'.**

A linear model predicts with a weighted sum: $\hat y = w_0 + w_1 x_1 + w_2 x_2 + \dots$. Each feature gets one weight, and that weight is the feature's effect regardless of the other features. So a plain linear model is structurally unable to say "the effect of $x_1$ changes depending on $x_2$".

       An interaction feature is simply the product of two features, $x_1 x_2$, added as a new column. Now the model can learn a weight $w_{12}$ on that product. Rewriting, the effect of $x_1$ becomes $w_1 + w_{12} x_2$ — it now depends on $x_2$. That is exactly the "the effect of location depends on age" behavior we wanted, and it is an AND-like combination: the product is large only when both features are large.

---

This notebook builds interaction features up one step at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

### Step 1 — Build an XOR dataset where the classes only matter together

There is no bundled XOR dataset, so we draw two features `x1` and `x2` from a normal distribution with a fixed seed. The label is `1` when the two features have the **same sign** (both positive or both negative) and `0` when they have opposite signs. That is the classic XOR pattern: the class depends on the *combination* of the two features, not on either one alone.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(0)          # fixed seed -> same numbers every run
n = 400

x1 = rng.normal(size=n)                 # feature 1
x2 = rng.normal(size=n)                 # feature 2
X_raw = np.column_stack([x1, x2])       # raw feature matrix, shape (n, 2)

# "same sign" -> class 1; "opposite sign" -> class 0. That is the XOR rule.
same_sign = np.sign(x1) == np.sign(x2)
y = same_sign.astype(int)

### Step 2 — Watch a linear model fail on the raw features

A plain linear model (`LogisticRegression`) can only draw one straight line. The XOR classes sit in the four quadrants — top-right and bottom-left are one class, the other two quadrants the other — and no single line can carve those apart. We fit on the raw `[x1, x2]` and the training accuracy sits near 0.5, barely better than a coin flip. The scatter on the left makes the four-quadrant pattern visible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left panel: scatter of the raw data, colored by class.
ax = axes[0]
ax.scatter(x1[y == 1], x2[y == 1], c="tab:blue",   s=18, label="y=1 (same sign)")
ax.scatter(x1[y == 0], x2[y == 0], c="tab:orange", s=18, label="y=0 (opp. sign)")
ax.axhline(0, color="gray", lw=0.8)
ax.axvline(0, color="gray", lw=0.8)
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("RAW data: XOR pattern\n(no straight line can split the 4 quadrants)")
ax.legend(loc="upper right", fontsize=8)

# Fit a linear model on the raw [x1, x2] -> straight-line boundary -> ~50%.
clf = LogisticRegression()
clf.fit(X_raw, y)
acc_raw = clf.score(X_raw, y)           # train accuracy on raw features
print(f"PROBLEM  -- linear model on raw [x1, x2]      accuracy = {acc_raw:.3f}")

### Step 3 — Add the interaction feature and refit

The product `x1 * x2` is the interaction feature. It is positive exactly when the two features share a sign, so the two classes land on opposite sides of zero — a single threshold separates them. The right panel plots `x1 * x2` against the class to show that clean split. We then refit the **same** `LogisticRegression` on `[x1, x2, x1*x2]`; the product does the heavy lifting and accuracy jumps to about 1.0.

In [ ]:
# The new feature: the product of the two raw features.
inter = (x1 * x2).reshape(-1, 1)             # shape (n, 1)
X_eng = np.column_stack([x1, x2, x1 * x2])   # raw features PLUS the product

# Right panel: show the interaction feature splits the classes at 0.
ax = axes[1]
# small random y-jitter only so overlapping points are visible (cosmetic).
jitter = rng.normal(scale=0.04, size=n)
ax.scatter(inter[y == 1, 0], y[y == 1] + jitter[y == 1],
           c="tab:blue",   s=18, label="y=1 (same sign)")
ax.scatter(inter[y == 0, 0], y[y == 0] + jitter[y == 0],
           c="tab:orange", s=18, label="y=0 (opp. sign)")
ax.axvline(0, color="red", lw=1.2, ls="--", label="threshold at x1*x2 = 0")
ax.set_xlabel("interaction feature  x1 * x2")
ax.set_ylabel("class  y")
ax.set_title("ENGINEERED feature: x1*x2 vs class\n(classes split cleanly at 0)")
ax.legend(loc="center right", fontsize=8)

plt.tight_layout()
plt.show()

# Re-fit the SAME model on [x1, x2, x1*x2] -> accuracy jumps to ~1.0.
clf2 = LogisticRegression()
clf2.fit(X_eng, y)
acc_eng = clf2.score(X_eng, y)
print(f"FIX      -- same model + interaction x1*x2    accuracy = {acc_eng:.3f}")

# Explicit before/after summary.
print(f"PROBLEM (raw): {acc_raw:.3f}   ->   FIX (engineered): {acc_eng:.3f}")

## Reference implementation — scikit-learn

### Step 1 — Load the Online News Popularity dataset

This is the book's example dataset (UCI Online News Popularity). We read the CSV, strip the leading spaces the file has in its headers, and split off the target. The target `shares` is how many times an article was shared; we drop the non-predictive `url` and `timedelta` columns and keep the rest as the base feature matrix of about 58 columns.

In [ ]:
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# Online News Popularity (UCI) -- from the book's GitHub:
#   github.com/alicezheng/feature-engineering-book  ->  data/OnlineNewsPopularity.csv
df = pd.read_csv("OnlineNewsPopularity.csv")
df.columns = [c.strip() for c in df.columns]   # the CSV has leading spaces in headers

# target = number of shares; drop the non-predictive columns
y = df["shares"].values
X = df.drop(columns=["url", "timedelta", "shares"]).values
print("base feature matrix:", X.shape)         # ~ (39644, 58)

### Step 2 — Expand to all pairwise interactions

`PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)` adds one product column for every unordered pair of features, keeping the originals but skipping the squared terms. With ~58 base columns this explodes to ~1711 columns — a quadratic blow-up that is the central cost of interaction features.

In [ ]:
# Add ALL pairwise interaction features (degree-2, no squared terms).
pf = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X2 = pf.fit_transform(X)
print("with interactions:  ", X2.shape)        # ~ (39644, 1711) -- quadratic blow-up

### Step 3 — Compare cross-validated R^2: base vs base + interactions

We score the same `LinearRegression` two ways with 5-fold cross-validation: on the base features alone, then on the base features plus all interactions. The book observes only a small improvement from the interactions — bought at the price of roughly 30x more columns and longer training time.

In [ ]:
lr = LinearRegression()

# Cross-validated R^2 with and without the interaction columns.
r2_base  = cross_val_score(lr, X,  y, cv=5, scoring="r2")
r2_inter = cross_val_score(lr, X2, y, cv=5, scoring="r2")

print("R^2 base features:        %.4f  (+/- %.4f)" % (r2_base.mean(),  r2_base.std()))
print("R^2 base + interactions:  %.4f  (+/- %.4f)" % (r2_inter.mean(), r2_inter.std()))
# The book observes a small improvement from the interaction features --
# bought at the cost of ~30x more columns and longer training time.

## Visualize the data & results

_On a real dataset, does adding pairwise interaction features lift a linear model's accuracy — and what does it cost in feature count?_

### Step 1 — Build base and interaction feature sets

We load the breast-cancer data and keep just the first 5 `mean` features so the interaction count stays small and readable. `PolynomialFeatures` then turns those 5 columns into 15: the 5 originals plus the C(5,2)=10 pairwise products.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

Xall, y = load_breast_cancer(return_X_y=True)  # 569 samples, 30 features
X = Xall[:, :5]                                 # the first 5 'mean' features
print("base:", X.shape)                         # (569, 5)

# add all pairwise interaction features (no squared terms)
pf = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X2 = pf.fit_transform(X)
print("with interactions:", X2.shape)           # (569, 15) = 5 + C(5,2)=10

### Step 2 — Standardize, then compare accuracy

Products of features can be on very different scales, so we standardize first (inside a pipeline, so the scaler is refit on each CV fold). We then compare 5-fold accuracy with and without the interactions. The lift is small but real here — about 0.9244 to 0.9279.

In [ ]:
# standardize first, then fit -- products are well-scaled this way
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))

acc_base  = cross_val_score(clf, X,  y, cv=5, scoring="accuracy").mean()
acc_inter = cross_val_score(clf, X2, y, cv=5, scoring="accuracy").mean()

print("acc base:               %.4f" % acc_base)    # ~0.9244
print("acc base+interactions:  %.4f" % acc_inter)   # ~0.9279  (small real lift)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You fit a plain LinearRegression to predict house price from two standardized features, location_score and buyer_age, and the model underfits: it predicts the same location premium for every buyer, but you know young and older buyers value neighborhoods differently. What single new feature lets a linear model express this, and what does its coefficient mean?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Name what the plain model can and cannot do. — _$\hat y = w_0 + w_1\,\text{loc} + w_2\,\text{age}$ gives location one fixed weight $w_1$ — the same premium for every age. It structurally cannot make the location effect vary with age._
- Form the interaction. — _Add the product column $\text{loc}\times\text{age}$. The model becomes $\hat y = w_0 + w_1\text{loc} + w_2\text{age} + w_{12}(\text{loc}\cdot\text{age})$._
- Read the new marginal effect. — _$\partial\hat y/\partial\,\text{loc} = w_1 + w_{12}\,\text{age}$ — the location premium now slides with buyer age, exactly the behavior you wanted._

**Answer:** Add the interaction feature $\text{location\_score} \times \text{buyer\_age}$ as a new column (e.g. via PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)). The linear model can then learn a weight $w_{12}$ on it, and the effect of location becomes $w_1 + w_{12}\cdot\text{buyer\_age}$ — no longer constant, but sliding with age. The coefficient $w_{12}$ says how much the location premium changes per unit of (standardized) buyer age: positive means older buyers value good locations more; negative means less. This is Zheng & Casari's "the effect of location depends on age" example, encoded as one product column. Because both features were standardized first, the product is well-scaled.

</details>

**Problem 2.** Your feature matrix has $d = 50$ numeric columns. You run PolynomialFeatures(degree=2, interaction_only=True, include_bias=False). How many columns come out, how many would the full polynomial give, and what two precautions does the book recommend?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count the interaction-only output. — _Interaction-only keeps the $d$ originals and adds one product per unordered pair: $\binom{50}{2}=\frac{50\cdot 49}{2}=1225$. Total $= 50 + 1225 = 1275$._
- Count the full polynomial. — _The full degree-2 expansion also adds the $d=50$ squared terms $x_i^2$: total $= 50 + 1225 + 50 = 1325$ (1275 features plus 50 squares)._
- Recall the trade-off. — _Both blow up as $O(d^2)$, so many columns risk overfitting and slow training — the book pairs interactions with regularization and feature selection._

**Answer:** Interaction-only: $50 + \binom{50}{2} = 50 + 1225 = \mathbf{1275}$ columns. Full polynomial (interaction_only=False): it also adds the 50 squared terms $x_i^2$, giving $1275 + 50 = \mathbf{1325}$. Both grow like $O(d^2)$, so 50 features became well over a thousand. The book's two precautions: (1) use regularization (ridge or lasso) so the extra columns cannot memorize noise, and (2) follow with feature selection to keep only the interactions that actually help. It also helps to standardize first so the products are well-scaled.

</details>

**Problem 3.** A colleague adds all pairwise interaction features to a random forest and to a logistic regression, expecting both to improve. One barely changes and may even get slightly worse; the other can improve. Which is which, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what each model can already represent. — _Trees split sequentially: a split on $x_1$ followed by a split on $x_2$ already encodes their interaction, so the forest learns interactions natively._
- Recall the linear model's limit. — _Logistic (a linear-in-parameters model) sums each feature times one weight and cannot represent a product unless you hand it the product column._
- Predict the outcome. — _Interaction features mostly add redundant, noisy columns to the forest (no gain, extra cost) but give the logistic model genuinely new expressive power._

**Answer:** The logistic regression can improve; the random forest barely changes (or slightly worsens). A linear/logistic model adds each feature times one weight and is structurally blind to feature products, so handing it $x_i x_j$ columns genuinely expands what it can fit. A random forest already learns interactions by splitting on one feature and then another, so the hand-built products are mostly redundant — they add dimensionality, training cost, and a little noise without new information, which can even nudge validation accuracy down. This is exactly the book's guidance: reserve interaction features for models that cannot learn interactions themselves (linear/logistic), not for trees or neural nets.

</details>